一、导入相关库

In [ ]:
import os
import sys
import time
import warnings
import numpy as np
import pandas as pd
import polars as pl
import alphalens as al
import matplotlib.pyplot as plt
from datetime import datetime

# 配置文件
try:
    import config_local as config
except ImportError:
    import config

# 导入数据接口sdk
import zenidatasdk as zd
client = zd.Client(
    base_url=config.ZENI_URL,
    username=config.ZENI_USERNAME,
    password=config.ZENI_PASSWORD, 
)

# 忽视警告信息
warnings.filterwarnings(action='ignore')

二、获取数据

In [ ]:
# 历史回测区间
init_date = '2018-01-01'
start_date = '2019-01-01'
end_date = '2025-09-30'
index_symbol = "000905.XSHG"

In [ ]:
# 获取指数成分股数据据
index_weights_df = client.get_index_constituents_df(
    index_symbol=index_symbol,
    start_date=start_date,
    end_date=end_date
)
index_weights_df = index_weights_df.rename(columns={"date": "datetime"})
symbols = index_weights_df["symbol"].unique().tolist()
index_weights_df

In [ ]:
# 获取日频bar数据
bars_1d_df = client.kline.get_kline_df(
    symbol=symbols,
    start_date=init_date,
    end_date=end_date,
    frequency="1d",
    adjust_type="pre",
    market="cn_stock"
)
bars_1d_df

In [ ]:
# 获取日频估值数据
fundamental_1d_df = client.get_valuation_df(
    symbols=symbols,
    start_date=start_date,
    end_date=end_date,
    fields="datetime,symbol,market_cap,circulating_market_cap,turnover_ratio,pe_ratio,pb_ratio,dividend_ratio"
)
fundamental_1d_df


# 构建市值数据 
# market_cap、circulating_market_cap
mkt_cap_name = "market_cap"
market_cap_df = fundamental_1d_df.set_index(["datetime", "symbol"])[[mkt_cap_name]]

# 负数和无穷值 & 对数处理
market_cap_df[mkt_cap_name] = np.where((market_cap_df[mkt_cap_name] <= 0) | (~np.isfinite(market_cap_df[mkt_cap_name])), 0, market_cap_df[mkt_cap_name])
market_cap_df[f"{mkt_cap_name}_log"] = np.log1p(market_cap_df[mkt_cap_name])
market_cap = market_cap_df[f"{mkt_cap_name}_log"]
market_cap

In [ ]:
# 获取行业数据
industry_constituents_df = client.get_industry_constituents_composite_df(
    symbols=symbols,
    category="sw_l1",
    start_date=start_date,
    end_date=end_date
)

# 构建双重索引的行业数据
industries = industry_constituents_df.set_index(["datetime", "symbol"])["industry_name"]
industries

三、构建价格数据

In [ ]:
# 多资产价格数据(开盘价买入)
prices_df = bars_1d_df[bars_1d_df["datetime"] >= start_date]
prices = prices_df.pivot_table(index="datetime", columns="symbol", values="open")
prices

四、构建因子变量

In [ ]:
# 计算因子数据
factors_df = ... # 因子函数 func()
# 因子值 shift 1 转换成实际使用时间(T+1)
factors_df["factor_value"] = factors_df.groupby('symbol')['factor_value'].transform(lambda x: x.shift(1))
factors_df

In [ ]:
# 与指数的交易日历、历史成分股数据对齐
factors_df = pd.merge(index_weights_df[["datetime", "symbol"]], factors_df, how="left", on=["datetime", "symbol"])
# 转换成[datetime, symbol]双重索引的factor_table
factors = factors_df.pivot_table(index=["datetime", "symbol"], columns="factor_name", values="factor_value")
factors

In [ ]:
factors.info()

五、合并因子与目标变量

In [ ]:
# 调用alphalens进行数据清洗
factor_data = al.utils.get_clean_factor_and_forward_returns(
    factor=factors,
    prices=prices,
    periods=(1, 5, 20),
    bins=None,
    quantiles=5,
    groupby=None,
    groupby_labels=None,
    binning_by_group=False,
    filter_zscore=20,
    max_loss=0.25,
    zero_aware=False,
    cumulative_returns=True
)
factor_data

六、alphalens因子测试

In [ ]:
# 调用alphalens进行因子评估
al.tears.create_full_tear_sheet(
    factor_data=factor_data,
    long_short=False,
    group_neutral=False,
    by_group=False
)